In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 02 - Interpolation Methods\n",
    "\n",
    "Compare different interpolation methods for handling missing values"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import os\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "sys.path.insert(0, os.path.join(os.getcwd(), '..'))\n",
    "\n",
    "from src.data_generator import ESGDataGenerator\n",
    "from src.interpolation import InterpolationHandler\n",
    "\n",
    "sns.set_style('whitegrid')\n",
    "plt.rcParams['figure.figsize'] = (14, 8)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Generate Data with Missing Values"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate data\n",
    "generator = ESGDataGenerator(n_periods=36, missing_rate=0.20, seed=42)\n",
    "df = generator.generate()\n",
    "\n",
    "print(f\"Original data shape: {df.shape}\")\n",
    "print(f\"\\nMissing values:\")\n",
    "print(df[['Environmental', 'Social', 'Governance']].isnull().sum())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Apply Different Interpolation Methods"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create interpolation handler\n",
    "handler = InterpolationHandler(df.copy())\n",
    "\n",
    "# Apply different methods\n",
    "df_linear = handler.interpolate(method='linear')\n",
    "df_cubic = handler.interpolate(method='cubic')\n",
    "df_poly = handler.interpolate(method='polynomial')\n",
    "\n",
    "print(\"\\nInterpolation complete for all methods!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Visualize Comparison"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Compare methods for Environmental parameter\n",
    "fig, axes = plt.subplots(2, 2, figsize=(16, 10))\n",
    "\n",
    "# Original with missing\n",
    "ax = axes[0, 0]\n",
    "ax.plot(df.index, df['Environmental'], marker='o', linestyle='-', color='gray', \n",
    "        label='Original', linewidth=2, markersize=5)\n",
    "missing_mask = df['Environmental'].isnull()\n",
    "ax.scatter(df[missing_mask].index, [df['Environmental'].mean()]*missing_mask.sum(),\n",
    "          color='red', s=100, marker='x', label='Missing', zorder=5)\n",
    "ax.set_title('Original Data (with Missing)', fontsize=12, fontweight='bold')\n",
    "ax.legend()\n",
    "ax.grid(True, alpha=0.3)\n",
    "\n",
    "# Linear interpolation\n",
    "ax = axes[0, 1]\n",
    "ax.plot(df_linear.index, df_linear['Environmental'], marker='o', linestyle='-', \n",
    "        color='#1f77b4', label='Linear', linewidth=2, markersize=5)\n",
    "ax.set_title('Linear Interpolation', fontsize=12, fontweight='bold')\n",
    "ax.legend()\n",
    "ax.grid(True, alpha=0.3)\n",
    "\n",
    "# Cubic interpolation\n",
    "ax = axes[1, 0]\n",
    "ax.plot(df_cubic.index, df_cubic['Environmental'], marker='o', linestyle='-',\n",
    "        color='#ff7f0e', label='Cubic', linewidth=2, markersize=5)\n",
    "ax.set_title('Cubic Spline Interpolation', fontsize=12, fontweight='bold')\n",
    "ax.legend()\n",
    "ax.grid(True, alpha=0.3)\n",
    "\n",
    "# Polynomial interpolation\n",
    "ax = axes[1, 1]\n",
    "ax.plot(df_poly.index, df_poly['Environmental'], marker='o', linestyle='-',\n",
    "        color='#2ca02c', label='Polynomial', linewidth=2, markersize=5)\n",
    "ax.set_title('Polynomial Interpolation', fontsize=12, fontweight='bold')\n",
    "ax.legend()\n",
    "ax.grid(True, alpha=0.3)\n",
    "\n",
    "plt.suptitle('Environmental Parameter - Interpolation Method Comparison', \n",
    "             fontsize=14, fontweight='bold', y=1.00)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Method Comparison Metrics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Compare methods\n",
    "comparison_data = {\n",
    "    'Method': ['Linear', 'Cubic', 'Polynomial'],\n",
    "    'Mean': [\n",
    "        df_linear['Environmental'].mean(),\n",
    "        df_cubic['Environmental'].mean(),\n",
    "        df_poly['Environmental'].mean()\n",
    "    ],\n",
    "    'Std': [\n",
    "        df_linear['Environmental'].std(),\n",
    "        df_cubic['Environmental'].std(),\n",
    "        df_poly['Environmental'].std()\n",
    "    ],\n",
    "    'Min': [\n",
    "        df_linear['Environmental'].min(),\n",
    "        df_cubic['Environmental'].min(),\n",
    "        df_poly['Environmental'].min()\n",
    "    ],\n",
    "    'Max': [\n",
    "        df_linear['Environmental'].max(),\n",
    "        df_cubic['Environmental'].max(),\n",
    "        df_poly['Environmental'].max()\n",
    "    ]\n",
    "}\n",
    "\n",
    "comparison_df = pd.DataFrame(comparison_data)\n",
    "print(\"\\nInterpolation Methods Comparison:\")\n",
    "print(comparison_df.to_string(index=False))\n",
    "\n",
    "# Visualization\n",
    "fig, axes = plt.subplots(1, 3, figsize=(15, 4))\n",
    "\n",
    "for idx, metric in enumerate(['Mean', 'Std', 'Max']):\n",
    "    ax = axes[idx]\n",
    "    comparison_df.plot(x='Method', y=metric, kind='bar', ax=ax, legend=False,\n",
    "                      color=['#1f77b4', '#ff7f0e', '#2ca02c'])\n",
    "    ax.set_title(f'{metric}', fontsize=12, fontweight='bold')\n",
    "    ax.set_ylabel(metric, fontsize=11)\n",
    "    ax.set_xlabel('')\n",
    "    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0)\n",
    "\n",
    "plt.suptitle('Comparison Metrics by Interpolation Method', fontsize=14, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}
